# IS455 Machine Learning App — Fraud Detection Pipeline

**Target:** `is_fraud` column in the `orders` table of `shop.db`  
**Methodology:** Cross-Industry Standard Process for Data Mining (CRISP-DM)  
**Chapters covered:** 2–4 (wrangling), 6 (exploration), 7 (pipelines), 8 (relationships), 13 (classification), 14 (ensembles), 15 (evaluation & tuning), 16 (feature selection), 17 (deployment)

---

## CRISP-DM Overview

| Phase | Description |
|-------|-------------|
| 1 | Business Understanding |
| 2 | Data Understanding |
| 3 | Data Preparation |
| 4 | Modeling |
| 5 | Evaluation & Tuning |
| 6 | Feature Selection |
| 7 | Deployment |


In [ ]:
# ── Global Imports ──────────────────────────────────────────────────────────
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
DB_PATH = 'shop.db'
MODEL_PATH = 'fraud_model.joblib'

print('All libraries loaded successfully.')

---
## Phase 1 — Business Understanding

### Problem Statement
The operations team needs to identify potentially fraudulent orders **before fulfillment** so that a human reviewer can verify them. A false negative (missed fraud) results in a direct financial loss. A false positive (flagging a legitimate order) causes customer friction and delays.

### Success Criteria
| Metric | Target |
|--------|--------|
| Recall (fraud caught) | ≥ 0.80 |
| Precision (flag accuracy) | ≥ 0.50 |
| ROC-AUC | ≥ 0.85 |

### Business Context
- The model must produce a **fraud probability score** for each order.
- Scores feed the web app's **priority queue**, surfacing the highest-risk orders first.
- The trained model is serialized and called by the **Run Scoring** action in the admin dashboard (Ch. 17).

### Cost Asymmetry
- **False Negative cost:** Full order value lost + chargeback fees (~$150 average).
- **False Positive cost:** Customer service time + possible order delay (~$5–10).
- ➔ We prioritize **recall** over precision; we will tune the decision threshold accordingly.


---
## Phase 2 — Data Understanding
*(Ch. 6: Feature-level exploration · Ch. 8: Discovering relationships)*


In [ ]:
# ── 2.1 Connect and Load ─────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

# Flatten the analytical view: orders + customers + order_items aggregate
query = '''
    SELECT
        o.order_id,
        o.customer_id,
        o.order_datetime,
        o.billing_zip,
        o.shipping_zip,
        o.shipping_state,
        o.payment_method,
        o.device_type,
        o.ip_country,
        o.promo_used,
        o.promo_code,
        o.order_subtotal,
        o.shipping_fee,
        o.tax_amount,
        o.order_total,
        o.risk_score,
        o.is_fraud,
        c.customer_segment,
        c.loyalty_tier,
        c.is_active,
        COALESCE(i.total_items,  0) AS total_items,
        COALESCE(i.avg_unit_price, 0) AS avg_unit_price,
        COALESCE(i.max_unit_price, 0) AS max_unit_price,
        COALESCE(i.distinct_products, 0) AS distinct_products
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN (
        SELECT
            order_id,
            SUM(quantity)            AS total_items,
            AVG(unit_price)          AS avg_unit_price,
            MAX(unit_price)          AS max_unit_price,
            COUNT(DISTINCT product_id) AS distinct_products
        FROM order_items
        GROUP BY order_id
    ) i ON o.order_id = i.order_id
'''

df_raw = pd.read_sql_query(query, conn)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# ── 2.2 Basic Info (Ch. 6 — Feature-level exploration) ───────────────────────
print('=== dtypes & non-null counts ===')
df_raw.info()
print('\n=== Descriptive statistics ===')
df_raw.describe().round(2)

In [ ]:
# ── 2.3 Target Variable Distribution ────────────────────────────────────────
target_counts = df_raw['is_fraud'].value_counts()
fraud_rate = df_raw['is_fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Legitimate (0)', 'Fraud (1)'], target_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Order Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Fraud Rate: {fraud_rate:.1f}%')

plt.suptitle('Target Variable: is_fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Class imbalance ratio — Legitimate : Fraud = {target_counts[0]/target_counts[1]:.1f} : 1')

In [ ]:
# ── 2.4 Missing Value Analysis (Ch. 6) ──────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0]

if missing_report.empty:
    print('No missing values detected after the JOIN.')
else:
    display(missing_report)

In [ ]:
# ── 2.5 Univariate Distributions of Key Numeric Features (Ch. 6) ─────────────
numeric_cols = ['order_subtotal', 'shipping_fee', 'order_total',
                'risk_score', 'total_items', 'avg_unit_price']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, col in enumerate(numeric_cols):
    df_raw[df_raw['is_fraud'] == 0][col].hist(ax=axes[i], bins=30, alpha=0.6,
                                               color='steelblue', label='Legit')
    df_raw[df_raw['is_fraud'] == 1][col].hist(ax=axes[i], bins=30, alpha=0.6,
                                               color='tomato', label='Fraud')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Numeric Feature Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6 Categorical Features vs. Fraud Rate (Ch. 8 — Relationships) ──────────
cat_cols = ['payment_method', 'device_type', 'customer_segment', 'loyalty_tier', 'ip_country']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    fraud_by_cat = df_raw.groupby(col)['is_fraud'].mean().sort_values(ascending=False)
    fraud_by_cat.plot(kind='bar', ax=axes[i], color='tomato', edgecolor='white')
    axes[i].set_title(f'Fraud Rate by {col}')
    axes[i].set_ylabel('Fraud Rate')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

axes[-1].set_visible(False)  # hide unused subplot
plt.suptitle('Fraud Rate by Categorical Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.7 Correlation Heatmap (Ch. 8) ─────────────────────────────────────────
corr_cols = numeric_cols + ['is_fraud', 'promo_used', 'is_active']
corr_matrix = df_raw[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix (Numeric Features + Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with is_fraud:')
print(corr_matrix['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False).round(4))

---
## Phase 3 — Data Preparation
*(Ch. 2–4: Wrangling & cleaning · Ch. 7: Automated preparation pipelines)*


In [ ]:
# ── 3.1 Feature Engineering (Ch. 4 — derived features) ──────────────────────
df = df_raw.copy()

# --- Temporal features ---
df['order_datetime'] = pd.to_datetime(df['order_datetime'], errors='coerce')
df['order_hour']       = df['order_datetime'].dt.hour
df['order_dayofweek']  = df['order_datetime'].dt.dayofweek   # 0=Mon, 6=Sun
df['is_weekend_order'] = (df['order_dayofweek'] >= 5).astype(int)
df['is_late_night']    = df['order_hour'].between(0, 5).astype(int)

# --- Address / geography mismatch flags ---
df['zip_mismatch']    = (df['billing_zip'] != df['shipping_zip']).astype(int)
df['foreign_ip']      = (df['ip_country'] != 'US').astype(int)

# --- High-value order flag (>95th percentile) ---
high_val_threshold    = df['order_total'].quantile(0.95)
df['high_value_order'] = (df['order_total'] > high_val_threshold).astype(int)

# --- Shipping ratio (shipping fee as a % of order total) ---
df['shipping_ratio']  = df['shipping_fee'] / df['order_total'].replace(0, np.nan)

print('Engineered features added:')
new_cols = ['order_hour', 'order_dayofweek', 'is_weekend_order', 'is_late_night',
            'zip_mismatch', 'foreign_ip', 'high_value_order', 'shipping_ratio']
print(df[new_cols].describe().round(3))

In [ ]:
# ── 3.2 Select Features and Target ──────────────────────────────────────────
NUMERIC_FEATURES = [
    'order_subtotal', 'shipping_fee', 'tax_amount', 'order_total',
    'risk_score', 'total_items', 'avg_unit_price', 'max_unit_price',
    'distinct_products', 'order_hour', 'order_dayofweek', 'shipping_ratio'
]

BINARY_FEATURES = [
    'promo_used', 'is_active', 'zip_mismatch', 'foreign_ip',
    'is_weekend_order', 'is_late_night', 'high_value_order'
]

CATEGORICAL_FEATURES = [
    'payment_method', 'device_type', 'customer_segment', 'loyalty_tier'
]

TARGET = 'is_fraud'

# Drop rows where the target is null (shouldn't be any, but defensive)
df = df.dropna(subset=[TARGET])

X = df[NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET].astype(int)

print(f'Feature matrix: {X.shape}  |  Target distribution:')
print(y.value_counts())

In [ ]:
# ── 3.3 Train / Test Split (stratified to preserve fraud rate) ───────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.3f}  |  Test fraud rate: {y_test.mean():.3f}')

In [ ]:
# ── 3.4 Automated Preprocessing Pipeline (Ch. 7 — sklearn Pipelines) ─────────
# Numeric: impute median → scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Binary flags: impute with 0 (no scaling needed, but keep consistent)
binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

# Categorical: impute mode → one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num',  numeric_transformer,     NUMERIC_FEATURES),
    ('bin',  binary_transformer,      BINARY_FEATURES),
    ('cat',  categorical_transformer, CATEGORICAL_FEATURES)
])

print('Preprocessing pipeline constructed.')
print(preprocessor)

---
## Phase 4 — Modeling
*(Ch. 13: Classification · Ch. 14: Ensemble methods)*

We train three models:
1. **Logistic Regression** — interpretable baseline (Ch. 13)
2. **Random Forest** — bagging ensemble (Ch. 14)
3. **Gradient Boosting** — boosting ensemble (Ch. 14)

All models use `class_weight='balanced'` to compensate for the ~6% fraud rate without resampling.


In [ ]:
# ── 4.1 Define Models ────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=4, subsample=0.8, random_state=RANDOM_STATE
    )
}

# Wrap each model in a full pipeline
pipelines = {
    name: Pipeline(steps=[('prep', preprocessor), ('clf', model)])
    for name, model in models.items()
}

print('Model pipelines defined:')
for name in pipelines:
    print(f'  • {name}')

In [ ]:
# ── 4.2 Train All Models ─────────────────────────────────────────────────────
trained_pipelines = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline
    print(f'  ✓  {name} trained.')

---
## Phase 5 — Evaluation, Model Selection & Tuning
*(Ch. 15: Evaluation, selection, and hyperparameter tuning)*


In [ ]:
# ── 5.1 Compute Test-Set Metrics for All Models ──────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, pipeline in trained_pipelines.items():
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    roc_auc  = roc_auc_score(y_test, y_proba)
    avg_prec = average_precision_score(y_test, y_proba)
    cv_roc   = cross_val_score(pipeline, X_train, y_train, cv=cv,
                                scoring='roc_auc', n_jobs=-1).mean()

    from sklearn.metrics import precision_score, recall_score, f1_score
    results.append({
        'Model':         name,
        'Accuracy':      accuracy_score(y_test, y_pred),
        'Precision':     precision_score(y_test, y_pred),
        'Recall':        recall_score(y_test, y_pred),
        'F1':            f1_score(y_test, y_pred),
        'ROC-AUC':       roc_auc,
        'Avg Precision': avg_prec,
        'CV ROC-AUC':    cv_roc
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
display(results_df.sort_values('ROC-AUC', ascending=False))

In [ ]:
# ── 5.2 ROC Curves & Precision-Recall Curves ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'seagreen']

for (name, pipeline), color in zip(trained_pipelines.items(), colors):
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)

    # Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.4,
                label=f'Baseline (={y_test.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle('Model Comparison — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Confusion Matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, pipeline) in zip(axes, trained_pipelines.items()):
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    ax.set_title(name)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 Select Best Model ────────────────────────────────────────────────────
best_model_name = results_df['ROC-AUC'].idxmax()
best_pipeline   = trained_pipelines[best_model_name]
print(f'Best model by ROC-AUC: {best_model_name}')
print()
y_pred_best = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=['Legitimate', 'Fraud']))

In [ ]:
# ── 5.5 Threshold Tuning (Ch. 15) ───────────────────────────────────────────
# Because the business cost of missing fraud is high, we find the threshold
# that maximizes recall while keeping precision >= 0.30.
y_proba_best = best_pipeline.predict_proba(X_test)[:, 1]
prec_vals, rec_vals, thresholds = precision_recall_curve(y_test, y_proba_best)

# Find the lowest threshold where precision >= 0.30
viable = [(t, p, r) for t, p, r in zip(thresholds, prec_vals[:-1], rec_vals[:-1]) if p >= 0.30]

if viable:
    # Among viable thresholds, pick one with maximum recall
    best_threshold, best_prec, best_rec = max(viable, key=lambda x: x[2])
else:
    best_threshold = 0.5
    best_prec, best_rec = 0, 0

print(f'Optimal decision threshold : {best_threshold:.3f}')
print(f'  Precision @ threshold    : {best_prec:.3f}')
print(f'  Recall    @ threshold    : {best_rec:.3f}')

# Apply the tuned threshold
y_pred_tuned = (y_proba_best >= best_threshold).astype(int)
print()
print('Classification report with tuned threshold:')
print(classification_report(y_test, y_pred_tuned, target_names=['Legitimate', 'Fraud']))

In [ ]:
# ── 5.6 Hyperparameter Tuning with GridSearchCV (Ch. 15) ─────────────────────
# Tune the best model (Random Forest parameters shown as example)
print(f'Tuning: {best_model_name}')

if 'Random Forest' in best_model_name:
    param_grid = {
        'clf__n_estimators': [100, 200],
        'clf__max_depth':    [8, 10, 15],
        'clf__min_samples_leaf': [3, 5]
    }
elif 'Gradient Boosting' in best_model_name:
    param_grid = {
        'clf__n_estimators':  [100, 200],
        'clf__learning_rate': [0.05, 0.10],
        'clf__max_depth':     [3, 5]
    }
else:
    param_grid = {
        'clf__C': [0.01, 0.1, 1.0, 10.0]
    }

grid_search = GridSearchCV(
    best_pipeline, param_grid,
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'Best params : {grid_search.best_params_}')
print(f'Best CV AUC : {grid_search.best_score_:.4f}')

tuned_pipeline = grid_search.best_estimator_

In [ ]:
# ── 5.7 Evaluate Tuned Model on Test Set ────────────────────────────────────
y_proba_tuned_model = tuned_pipeline.predict_proba(X_test)[:, 1]
y_pred_tuned_model  = (y_proba_tuned_model >= best_threshold).astype(int)

print('Tuned model — test-set metrics:')
print(f'  ROC-AUC   : {roc_auc_score(y_test, y_proba_tuned_model):.4f}')
print(f'  Avg Prec  : {average_precision_score(y_test, y_proba_tuned_model):.4f}')
print()
print(classification_report(y_test, y_pred_tuned_model, target_names=['Legitimate', 'Fraud']))

---
## Phase 6 — Feature Selection
*(Ch. 16: Feature selection methods)*


In [ ]:
# ── 6.1 Extract Feature Names After Preprocessing ───────────────────────────
prep_fitted = tuned_pipeline.named_steps['prep']

ohe_feature_names = (
    prep_fitted
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(CATEGORICAL_FEATURES)
    .tolist()
)

all_feature_names = NUMERIC_FEATURES + BINARY_FEATURES + ohe_feature_names
print(f'Total features after encoding: {len(all_feature_names)}')

In [ ]:
# ── 6.2 Feature Importance Plot ──────────────────────────────────────────────
clf_step = tuned_pipeline.named_steps['clf']

if hasattr(clf_step, 'feature_importances_'):
    importances = clf_step.feature_importances_
    feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    feat_imp.head(20).plot(kind='barh', color='steelblue', edgecolor='white')
    plt.gca().invert_yaxis()
    plt.title('Top-20 Feature Importances (Tuned Model)', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()
else:
    # Logistic Regression — use absolute coefficients
    coefs = np.abs(clf_step.coef_[0])
    feat_imp = pd.Series(coefs, index=all_feature_names).sort_values(ascending=False)
    feat_imp.head(20).plot(kind='barh', color='steelblue')
    plt.title('Top-20 Logistic Regression |Coefficients|')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 6.3 SelectFromModel — Reduce to High-Importance Features ─────────────────
# Build a fresh pipeline that includes a feature selection step
selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE
    ),
    threshold='mean'  # keep features with importance above the mean
)

pipeline_with_selection = Pipeline(steps=[
    ('prep',     preprocessor),
    ('selector', selector),
    ('clf',      RandomForestClassifier(
                     n_estimators=200, class_weight='balanced',
                     random_state=RANDOM_STATE))
])

pipeline_with_selection.fit(X_train, y_train)

# How many features remain?
n_selected = pipeline_with_selection.named_steps['selector'].get_support().sum()
print(f'Features after SelectFromModel: {n_selected} / {len(all_feature_names)}')

# Evaluate on test set
y_proba_sel = pipeline_with_selection.predict_proba(X_test)[:, 1]
auc_sel = roc_auc_score(y_test, y_proba_sel)
print(f'ROC-AUC with reduced feature set: {auc_sel:.4f}')
print(f'ROC-AUC with full feature set   : {roc_auc_score(y_test, y_proba_tuned_model):.4f}')

In [ ]:
# ── 6.4 Top Selected Features ────────────────────────────────────────────────
selector_fitted = pipeline_with_selection.named_steps['selector']
support_mask    = selector_fitted.get_support()
selected_names  = [name for name, sel in zip(all_feature_names, support_mask) if sel]

print('Selected features:')
for i, name in enumerate(selected_names, 1):
    print(f'  {i:2d}. {name}')

---
## Phase 7 — Deployment
*(Ch. 17: Model serialization and integration into the web app pipeline)*

The trained model is serialized to `fraud_model.joblib`. The web app's **Run Scoring** action calls `score_orders()` from `score.py`, which:
1. Loads the artifact.
2. Fetches unscored orders from the database.
3. Engineers the same features used in training.
4. Writes `fraud_probability` and `predicted_fraud` to the `order_predictions` table.


In [ ]:
# ── 7.1 Package Artifact ─────────────────────────────────────────────────────
artifact = {
    'pipeline':           tuned_pipeline,       # full prep + clf pipeline
    'decision_threshold': best_threshold,        # tuned for high recall
    'numeric_features':   NUMERIC_FEATURES,
    'binary_features':    BINARY_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'all_input_columns':  NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES,
    'training_roc_auc':   roc_auc_score(y_test, tuned_pipeline.predict_proba(X_test)[:, 1])
}

joblib.dump(artifact, MODEL_PATH)
print(f'Artifact saved to {MODEL_PATH}')
print(f'  Decision threshold : {best_threshold:.4f}')
print(f'  Training ROC-AUC   : {artifact["training_roc_auc"]:.4f}')

In [ ]:
# ── 7.2 Validate Schema — order_predictions table ────────────────────────────
# The web app reads fraud scores from order_predictions.
# Verify the schema includes the fraud columns.
cursor = conn.cursor()
cursor.execute('PRAGMA table_info(order_predictions);')
schema_rows = cursor.fetchall()

print('Current order_predictions schema:')
for row in schema_rows:
    print(f'  col {row[0]}: {row[1]} ({row[2]})')

existing_cols = [r[1] for r in schema_rows]
print()

# Add fraud columns if they do not exist (idempotent migration)
fraud_columns = {
    'fraud_probability':  'REAL',
    'predicted_fraud':    'INTEGER'
}
for col_name, col_type in fraud_columns.items():
    if col_name not in existing_cols:
        conn.execute(f'ALTER TABLE order_predictions ADD COLUMN {col_name} {col_type}')
        print(f'  [+] Added column: {col_name} ({col_type})')
    else:
        print(f'  [✓] Column already exists: {col_name}')

conn.commit()
print('\nSchema migration complete.')

In [ ]:
# ── 7.3 Inference Demo — Simulating the Scoring Action ───────────────────────
# This mirrors the logic in score.py that the web app calls.

def engineer_features(df_orders: pd.DataFrame) -> pd.DataFrame:
    """Apply the same feature engineering used during training."""
    df = df_orders.copy()
    df['order_datetime']  = pd.to_datetime(df['order_datetime'], errors='coerce')
    df['order_hour']      = df['order_datetime'].dt.hour
    df['order_dayofweek'] = df['order_datetime'].dt.dayofweek
    df['is_weekend_order']= (df['order_dayofweek'] >= 5).astype(int)
    df['is_late_night']   = df['order_hour'].between(0, 5).astype(int)
    df['zip_mismatch']    = (df['billing_zip'] != df['shipping_zip']).astype(int)
    df['foreign_ip']      = (df['ip_country'] != 'US').astype(int)
    df['high_value_order']= (df['order_total'] > high_val_threshold).astype(int)
    df['shipping_ratio']  = df['shipping_fee'] / df['order_total'].replace(0, np.nan)
    return df


def score_orders(db_path: str = DB_PATH, model_path: str = MODEL_PATH):
    """Load model, score all orders, write results to order_predictions."""
    artifact = joblib.load(model_path)
    pipeline  = artifact['pipeline']
    threshold = artifact['decision_threshold']
    all_cols  = artifact['all_input_columns']

    conn = sqlite3.connect(db_path)
    orders_df = pd.read_sql_query('''
        SELECT
            o.order_id,
            o.order_datetime, o.billing_zip, o.shipping_zip,
            o.payment_method, o.device_type, o.ip_country,
            o.promo_used, o.order_subtotal, o.shipping_fee,
            o.tax_amount, o.order_total, o.risk_score,
            c.customer_segment, c.loyalty_tier, c.is_active,
            COALESCE(i.total_items, 0)       AS total_items,
            COALESCE(i.avg_unit_price, 0)    AS avg_unit_price,
            COALESCE(i.max_unit_price, 0)    AS max_unit_price,
            COALESCE(i.distinct_products, 0) AS distinct_products
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        LEFT JOIN (
            SELECT order_id,
                   SUM(quantity)              AS total_items,
                   AVG(unit_price)            AS avg_unit_price,
                   MAX(unit_price)            AS max_unit_price,
                   COUNT(DISTINCT product_id) AS distinct_products
            FROM order_items GROUP BY order_id
        ) i ON o.order_id = i.order_id
    ''', conn)

    orders_df = engineer_features(orders_df)
    X_score   = orders_df[all_cols]

    fraud_proba   = pipeline.predict_proba(X_score)[:, 1]
    predicted_fraud = (fraud_proba >= threshold).astype(int)

    from datetime import datetime, timezone
    now = datetime.now(timezone.utc).isoformat()

    # Upsert predictions
    cursor = conn.cursor()
    for order_id, prob, pred in zip(orders_df['order_id'], fraud_proba, predicted_fraud):
        cursor.execute('''
            INSERT INTO order_predictions
                (order_id, fraud_probability, predicted_fraud, prediction_timestamp)
            VALUES (?, ?, ?, ?)
            ON CONFLICT(order_id) DO UPDATE SET
                fraud_probability    = excluded.fraud_probability,
                predicted_fraud      = excluded.predicted_fraud,
                prediction_timestamp = excluded.prediction_timestamp
        ''', (int(order_id), float(prob), int(pred), now))

    conn.commit()
    conn.close()
    print(f'Scored {len(orders_df)} orders.  '
          f'Flagged as fraud: {int(predicted_fraud.sum())} '
          f'({predicted_fraud.mean()*100:.1f}%)')
    return fraud_proba, predicted_fraud


# Run scoring demo
proba_out, pred_out = score_orders()

# Show sample predictions
sample = pd.DataFrame({
    'order_id':          df['order_id'].values[:10],
    'fraud_probability': proba_out[:10].round(4),
    'predicted_fraud':   pred_out[:10],
    'actual_is_fraud':   df['is_fraud'].values[:10]
})
print()
display(sample)

In [ ]:
# ── 7.4 Priority Queue — Top 20 Highest-Risk Orders ─────────────────────────
# This is what the web app's admin dashboard surfaces after Run Scoring.
priority_queue = pd.read_sql_query('''
    SELECT
        op.order_id,
        ROUND(op.fraud_probability, 4) AS fraud_probability,
        op.predicted_fraud,
        o.order_total,
        o.payment_method,
        o.is_fraud AS ground_truth
    FROM order_predictions op
    JOIN orders o ON op.order_id = o.order_id
    WHERE op.predicted_fraud = 1
    ORDER BY op.fraud_probability DESC
    LIMIT 20
''', conn)

print(f'Priority queue — {len(priority_queue)} orders flagged for review:')
display(priority_queue)

conn.close()

In [ ]:
# ── 7.5 Final Summary ────────────────────────────────────────────────────────
loaded_artifact = joblib.load(MODEL_PATH)

print('=' * 55)
print('  FRAUD DETECTION MODEL — DEPLOYMENT SUMMARY')
print('=' * 55)
print(f'  Model artifact  : {MODEL_PATH}')
print(f'  Best model      : {best_model_name}')
print(f'  Decision thresh : {loaded_artifact["decision_threshold"]:.4f}')
print(f'  Training ROC-AUC: {loaded_artifact["training_roc_auc"]:.4f}')
print(f'  Input features  : {len(loaded_artifact["all_input_columns"])}')
print('=' * 55)
print()
print('CRISP-DM phases completed:')
phases = [
    ('Business Understanding', 'Fraud detection problem & success criteria defined'),
    ('Data Understanding',     'SQLite load, univariate & relationship exploration (Ch.6,8)'),
    ('Data Preparation',       'Feature engineering, imputation, encoding pipeline (Ch.2-4,7)'),
    ('Modeling',               'Logistic Regression, Random Forest, Gradient Boosting (Ch.13,14)'),
    ('Evaluation & Tuning',    'ROC-AUC, PR curve, confusion matrix, GridSearchCV (Ch.15)'),
    ('Feature Selection',      'Importance ranking, SelectFromModel (Ch.16)'),
    ('Deployment',             'joblib serialization, score_orders() integration (Ch.17)'),
]
for phase, detail in phases:
    print(f'  ✓  {phase:<28} {detail}')